In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully")

Libraries imported successfully


In [61]:
"""
Configure project, dataset, and output directories.
DATA_DIR selects the dataset (default: ./data/).
"""

import os
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Dataset directory
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).resolve()

# Data layers
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Analysis outputs
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHARTS_DIR = OUTPUT_DIR / "charts"
METRICS_DIR = OUTPUT_DIR / "metrics"

# Create output directories
for directory in [OUTPUT_DIR, CHARTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: d:\Projects\remarcable-assignment
Data directory: D:\Projects\remarcable-assignment\data


In [28]:
purchase_orders = pd.read_csv(
    SILVER_DIR / "purchase_orders.csv"
)

itemized_purchase_orders = pd.read_csv(
    SILVER_DIR / "itemized_purchase_orders.csv"
)

items = pd.read_csv(
    SILVER_DIR / "items.csv"
)

item_pricing = pd.read_csv(
    SILVER_DIR / "item_pricing.csv"
)

datasets = {
    "purchase_orders": purchase_orders,
    "itemized_purchase_orders": itemized_purchase_orders,
    "items": items,
    "item_pricing": item_pricing
}

In [29]:
overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2
    }
    for name, df in datasets.items()
])

display(overview)

,dataset,rows,columns,memory_mb
0,purchase_orders,3000,7,1.18
1,itemized_purchase_orders,8483,5,1.70
2,items,358,6,0.14
3,item_pricing,154764,4,30.11


In [30]:
# Convert date columns to datetime
date_columns = {
    "purchase_orders": ["order_date"],
    "item_pricing": ["price_effective_date"]
}

for dataset, columns in date_columns.items():
    for column in columns:
        datasets[dataset][column] = pd.to_datetime(
            datasets[dataset][column], errors="coerce"
        )

purchase_orders = datasets["purchase_orders"]
item_pricing = datasets["item_pricing"]

In [31]:
# Profile missing values across all datasets
missing_results = []

for name, df in datasets.items():
    for column in df.columns:
        missing_results.append({
            "dataset": name,
            "column": column,
            "missing_count": df[column].isna().sum(),
            "missing_pct": df[column].isna().mean() * 100
        })

missing_df = pd.DataFrame(missing_results).sort_values(
    "missing_pct", ascending=False
)

display(missing_df)

missing_df.to_csv(METRICS_DIR / "missing_values.csv", index=False)

,dataset,column,missing_count,missing_pct
13,items,description,2,0.56
11,itemized_purchase_orders,unit_price_paid,26,0.31
14,items,category,1,0.28
16,items,manufacturer_id,1,0.28
17,items,manufacturer_name,1,0.28
0,purchase_orders,po_number,0,0.00
5,purchase_orders,status,0,0.00
4,purchase_orders,order_date,0,0.00
3,purchase_orders,vendor_name,0,0.00
2,purchase_orders,vendor_id,0,0.00


In [32]:
dim_customer = pd.read_csv(
    GOLD_DIR / "dim_customer.csv"
)

dim_item = pd.read_csv(
    GOLD_DIR / "dim_item.csv"
)

dim_vendor = pd.read_csv(
    GOLD_DIR / "dim_vendor.csv"
)

fct_purchase_order_line = pd.read_csv(
    GOLD_DIR / "fct_purchase_order_line.csv"
)

item_purchase_history = pd.read_csv(
    GOLD_DIR / "item_purchase_history.csv"
)

In [33]:
# Compare Silver and Gold purchase order lines
silver_lines = pd.read_csv(SILVER_DIR / "itemized_purchase_orders.csv")
gold_lines = pd.read_csv(GOLD_DIR / "fct_purchase_order_line.csv")

print("Silver rows:", len(silver_lines))
print("Gold rows:", len(gold_lines))

Silver rows: 8483
Gold rows: 8475


In [34]:
# Compare Silver and Gold column structures
print("SILVER COLUMNS")
print(silver_lines.columns.tolist())

print("\nGOLD COLUMNS")
print(gold_lines.columns.tolist())

SILVER COLUMNS
['po_number', 'customer_id', 'item_id', 'quantity_ordered', 'unit_price_paid']

GOLD COLUMNS
['po_number', 'customer_id', 'vendor_id', 'order_date', 'status', 'item_id', 'quantity_ordered', 'unit_price_paid', 'extended_amount']


In [35]:
# Check duplicate business keys
key_columns = ["po_number", "customer_id", "item_id"]

print("Silver duplicate business keys:",
      silver_lines.duplicated(subset=key_columns).sum())

print("Gold duplicate business keys:",
      gold_lines.duplicated(subset=key_columns).sum())

Silver duplicate business keys: 166
Gold duplicate business keys: 166


In [36]:
# Identify Silver lines excluded from the validated Gold fact
comparison_keys = ["po_number", "customer_id", "item_id"]

silver_only = (
    silver_lines[comparison_keys].drop_duplicates()
    .merge(
        gold_lines[comparison_keys].drop_duplicates(),
        on=comparison_keys,
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

print("Missing Silver business keys:", len(silver_only))
display(silver_only)

Missing Silver business keys: 8


,po_number,customer_id,item_id
2423,PO-100930,CUST-H,ITM-0246
2530,PO-100971,CUST-C,ITM-0431
3864,PO-101496,CUST-E,ITM-0914
5405,PO-102076,CUST-A,ITM-0414
6586,PO-102545,CUST-G,ITM-0629
6737,PO-102600,CUST-F,ITM-0733
7451,PO-102865,CUST-D,ITM-0609
7784,PO-102989,CUST-F,ITM-0805


In [37]:
# Reconcile PO lines against headers using PO number
joined = itemized_purchase_orders.merge(
    purchase_orders,
    on="po_number",
    how="left",
    indicator=True,
    suffixes=("_line", "_header")
)

matched = joined[
    (joined["_merge"] == "both") &
    (joined["customer_id_line"] == joined["customer_id_header"])
]

customer_mismatches = joined[
    (joined["_merge"] == "both") &
    (joined["customer_id_line"] != joined["customer_id_header"])
]

unmatched = joined[joined["_merge"] == "left_only"]

print("Itemized purchase orders:", len(itemized_purchase_orders))
print("Purchase orders:", len(purchase_orders))
print("Matched:", len(matched))
print("Customer mismatches:", len(customer_mismatches))
print("Unmatched PO numbers:", len(unmatched))

display(customer_mismatches)

Itemized purchase orders: 8483
Purchase orders: 3000
Matched: 8462
Customer mismatches: 13
Unmatched PO numbers: 8


,po_number,customer_id_line,item_id,quantity_ordered,unit_price_paid,customer_id_header,vendor_id,vendor_name,order_date,status,total_amount,_merge
878,PO-100315,CUST-C,ITM-1014,6,2.50,CUST-G,VEND-10,Ferrowatt Industrial Supply,2024-03-11,Received,51.96,both
1103,PO-100402,CUST-H,ITM-0527,4,8.44,CUST-A,VEND-07,BrightPath Electrical Supply,2025-11-17,Received,184.02,both
3561,PO-101343,CUST-C,ITM-0610,6,2.23,CUST-A,VEND-03,Circuit Point Supply,2026-03-24,Received,13.38,both
3717,PO-101404,CUST-G,ITM-0402,1,2.20,CUST-E,VEND-09,Current Line Supply,2026-03-25,Received,83.38,both
4024,PO-101527,CUST-D,ITM-0610,6,2.32,CUST-A,VEND-10,Ferrowatt Industrial Supply,2025-12-21,Received,86.10,both
4316,PO-101629,CUST-F,ITM-0230,5,2.97,CUST-B,VEND-12,Voltec Supply Group,2024-11-08,Received,17.65,both
4888,PO-101844,CUST-G,ITM-0225,2,2.46,CUST-C,VEND-02,Voltway Electrical Distributors,2024-07-31,Closed,53.56,both
5353,PO-102013,CUST-C,ITM-0430,10,0.50,CUST-E,VEND-08,Wattstone Distributors,2026-06-22,Open,114.18,both
5384,PO-102022,CUST-D,ITM-0306,24,0.90,CUST-C,VEND-03,Circuit Point Supply,2024-11-06,Closed,58.26,both
6192,PO-102339,CUST-A,ITM-0808,1,1.92,CUST-G,VEND-10,Ferrowatt Industrial Supply,2026-03-19,Received,13.20,both


In [38]:
# Identify duplicate PO headers
duplicate_po = purchase_orders[
    purchase_orders.duplicated(subset=["po_number"], keep=False)
].sort_values(["po_number", "customer_id"])

print("Duplicate PO headers:", len(duplicate_po))
display(duplicate_po)

Duplicate PO headers: 0


,po_number,customer_id,vendor_id,vendor_name,order_date,status,total_amount


In [39]:
# Identify PO numbers associated with multiple customers
duplicate_po_summary = (
    duplicate_po.groupby("po_number")
    .agg(
        customer_count=("customer_id", "nunique"),
        customers=("customer_id", lambda x: list(x.unique()))
    )
    .reset_index()
)

duplicate_po_summary = duplicate_po_summary[
    duplicate_po_summary["customer_count"] > 1
]

display(duplicate_po_summary)

,po_number,customer_count,customers


In [40]:
# Assess PO header uniqueness
print("Total PO headers:", len(purchase_orders))
print("Unique PO numbers:", purchase_orders["po_number"].nunique())
print(
    "Unique PO + Customer combinations:",
    purchase_orders[["po_number", "customer_id"]].drop_duplicates().shape[0]
)

Total PO headers: 3000
Unique PO numbers: 3000
Unique PO + Customer combinations: 3000


In [42]:
# Inspect unmatched purchase order lines
display(
    unmatched[
        ["po_number", "customer_id_line", "item_id",
         "quantity_ordered", "unit_price_paid"]
    ].sort_values(["po_number", "customer_id_line"])
)

,po_number,customer_id_line,item_id,quantity_ordered,unit_price_paid
2483,PO-100930,CUST-H,ITM-0246,2,2.07
2594,PO-100971,CUST-C,ITM-0431,4,0.78
3949,PO-101496,CUST-E,ITM-0914,4,1.53
5517,PO-102076,CUST-A,ITM-0414,4,1.87
6724,PO-102545,CUST-G,ITM-0629,2,3.00
6877,PO-102600,CUST-F,ITM-0733,1,22.21
7605,PO-102865,CUST-D,ITM-0609,2,1.60
7942,PO-102989,CUST-F,ITM-0805,6,32.21


In [43]:
# Classify unmatched lines by reconciliation issue
po_lookup = purchase_orders[["po_number", "customer_id"]].drop_duplicates()

unmatched_analysis = unmatched[
    ["po_number", "customer_id_line", "item_id"]
].merge(
    po_lookup,
    on="po_number",
    how="left",
    suffixes=("_line", "_header"),
    indicator="po_match_status"
)

unmatched_analysis["issue"] = np.where(
    unmatched_analysis["po_match_status"] == "left_only",
    "PO does not exist in headers",
    "Customer ID mismatch"
)

display(unmatched_analysis)

,po_number,customer_id_line,item_id,customer_id,po_match_status,issue
0,PO-100930,CUST-H,ITM-0246,NaN,left_only,PO does not exist in headers
1,PO-100971,CUST-C,ITM-0431,NaN,left_only,PO does not exist in headers
2,PO-101496,CUST-E,ITM-0914,NaN,left_only,PO does not exist in headers
3,PO-102076,CUST-A,ITM-0414,NaN,left_only,PO does not exist in headers
4,PO-102545,CUST-G,ITM-0629,NaN,left_only,PO does not exist in headers
5,PO-102600,CUST-F,ITM-0733,NaN,left_only,PO does not exist in headers
6,PO-102865,CUST-D,ITM-0609,NaN,left_only,PO does not exist in headers
7,PO-102989,CUST-F,ITM-0805,NaN,left_only,PO does not exist in headers


In [46]:
# Inspect customer attribution mismatches
customer_mismatches = unmatched_analysis[
    unmatched_analysis["issue"] == "Customer ID mismatch"
]

display(customer_mismatches)

,po_number,customer_id_line,item_id,customer_id,po_match_status,issue


In [21]:
display(
    itemized_purchase_orders[
        itemized_purchase_orders["po_number"] == "PO-102339"
    ]
)

display(
    purchase_orders[
        purchase_orders["po_number"] == "PO-102339"
    ]
)

,po_number,customer_id,item_id,quantity_ordered,unit_price_paid
6192,PO-102339,CUST-A,ITM-0808,1,1.92
6193,PO-102339,CUST-G,ITM-1014,4,2.82


,po_number,customer_id,vendor_id,vendor_name,order_date,status,total_amount
2206,PO-102339,CUST-G,VEND-10,Ferrowatt Industrial Supply,2026-03-19,Received,13.20


In [47]:
items[
    items["item_id"] == "ITM-0808"
]

,item_id,description,category,unit_of_measure,manufacturer_id,manufacturer_name
283,ITM-0808,Ground Rod Clamp 1/2in,Grounding,EA,MFG-02,Coppermark Cable Corp.


In [48]:
display(
    fct_purchase_order_line[
        fct_purchase_order_line["po_number"] == "PO-102339"
    ]
)

,po_number,customer_id,vendor_id,order_date,status,item_id,quantity_ordered,unit_price_paid,extended_amount
6188,PO-102339,CUST-A,VEND-10,2026-03-19,Received,ITM-0808,1,1.92,1.92
6189,PO-102339,CUST-G,VEND-10,2026-03-19,Received,ITM-1014,4,2.82,11.28


In [50]:
display(customer_mismatches)

,po_number,customer_id_line,item_id,customer_id,po_match_status,issue


In [ ]:
# Identify and summarize customer attribution mismatches
po_customer_check = itemized_purchase_orders.merge(
    purchase_orders[["po_number", "customer_id"]],
    on="po_number",
    how="inner",
    suffixes=("_line", "_header")
)

mismatches = po_customer_check[
    po_customer_check["customer_id_line"]
    != po_customer_check["customer_id_header"]
]

print("Customer mismatches:", len(mismatches))

display(
    mismatches.groupby(
        ["customer_id_line", "customer_id_header"]
    ).size().reset_index(name="records")
)

Customer mismatches: 13


,customer_id_line,customer_id_header,records
0,CUST-A,CUST-G,1
1,CUST-B,CUST-D,1
2,CUST-C,CUST-A,1
3,CUST-C,CUST-E,1
4,CUST-C,CUST-G,1
5,CUST-D,CUST-A,1
6,CUST-D,CUST-C,1
7,CUST-E,CUST-H,1
8,CUST-F,CUST-B,1
9,CUST-G,CUST-A,1


In [54]:
print("Total purchase lines:", len(itemized_purchase_orders))
print("Customer mismatches:", len(mismatches))
print("Missing PO headers:", len(unmatched_analysis[
    unmatched_analysis["issue"] == "PO does not exist in headers"
]))

print(
    "Company A affected lines:",
    len(mismatches[
        mismatches["customer_id_line"] == "CUST-A"
    ])
)

Total purchase lines: 8483
Customer mismatches: 13
Missing PO headers: 8
Company A affected lines: 1


In [55]:
# Check PO line references against item master
item_check = itemized_purchase_orders.merge(
    items[["item_id"]],
    on="item_id",
    how="left",
    indicator="item_match"
)

print(item_check["item_match"].value_counts())

item_match
both          8473
left_only       10
right_only       0
Name: count, dtype: int64


In [56]:
print("Zero or negative quantities:",
      (itemized_purchase_orders["quantity_ordered"] <= 0).sum())

print("Zero or negative unit prices:",
      (itemized_purchase_orders["unit_price_paid"] <= 0).sum())

Zero or negative quantities: 0
Zero or negative unit prices: 0


In [57]:
# Inspect PO lines referencing missing items
missing_items = item_check[item_check["item_match"] == "left_only"]

print("Missing item references:", len(missing_items))

display(
    missing_items[
        ["po_number", "customer_id", "item_id",
         "quantity_ordered", "unit_price_paid"]
    ]
)

Missing item references: 10


,po_number,customer_id,item_id,quantity_ordered,unit_price_paid
182,PO-100071,CUST-H,ITM-0822,2,33.71
2288,PO-100857,CUST-A,ITM-0240,4,44.09
3223,PO-101198,CUST-C,ITM-0116,1,20.42
3575,PO-101349,CUST-A,ITM-0305,2,41.17
4785,PO-101807,CUST-C,ITM-1002,6,17.28
4841,PO-101829,CUST-B,ITM-0239,2,56.03
7306,PO-102748,CUST-C,ITM-0701,6,26.71
7893,PO-102971,CUST-B,ITM-0935,6,40.58
8289,PO-103117,CUST-E,ITM-0916,4,13.33
8394,PO-103154,CUST-E,ITM-0417,4,53.98


In [58]:
data_quality_summary = pd.DataFrame([
    {
        "check": "Total purchase lines",
        "count": len(itemized_purchase_orders),
        "status": "INFO"
    },
    {
        "check": "Customer ID mismatches",
        "count": len(mismatches),
        "status": "WARNING"
    },
    {
        "check": "Missing PO headers",
        "count": len(unmatched_analysis[
            unmatched_analysis["issue"] == "PO does not exist in headers"
        ]),
        "status": "WARNING"
    },
    {
        "check": "Missing item references",
        "count": len(missing_items),
        "status": "WARNING"
    },
    {
        "check": "Invalid quantities",
        "count": (itemized_purchase_orders["quantity_ordered"] <= 0).sum(),
        "status": "PASS"
    },
    {
        "check": "Invalid unit prices",
        "count": (itemized_purchase_orders["unit_price_paid"] <= 0).sum(),
        "status": "PASS"
    }
])

display(data_quality_summary)

data_quality_summary.to_csv(
    OUTPUT_DIR / "metrics" / "data_quality_summary.csv",
    index=False
)

,check,count,status
0,Total purchase lines,8483,INFO
1,Customer ID mismatches,13,WARNING
2,Missing PO headers,8,WARNING
3,Missing item references,10,WARNING
4,Invalid quantities,0,PASS
5,Invalid unit prices,0,PASS


In [59]:
# Join purchase order headers and lines
reconciled = itemized_purchase_orders.merge(
    purchase_orders,
    on="po_number",
    how="inner",
    suffixes=("_line", "_header")
)

# Separate mismatches
customer_mismatches = reconciled[
    reconciled["customer_id_line"] !=
    reconciled["customer_id_header"]
].copy()

# Retain matching records for Gold
validated_lines = reconciled[
    reconciled["customer_id_line"] ==
    reconciled["customer_id_header"]
].copy()

# Rename line-level customer ID
validated_lines = validated_lines.rename(
    columns={"customer_id_line": "customer_id"}
)

# Calculate line-level expenditure
validated_lines["extended_amount"] = (
    validated_lines["quantity_ordered"] *
    validated_lines["unit_price_paid"]
)

validated_lines = validated_lines[["po_number", "customer_id", "vendor_id", "order_date", "status", 
                                   "item_id", "quantity_ordered", "unit_price_paid", "extended_amount"]]

print("Total joined lines:", len(reconciled))
print("Mismatched lines:", len(customer_mismatches))
print("Validated lines:", len(validated_lines))

Total joined lines: 8475
Mismatched lines: 13
Validated lines: 8462


In [60]:
display(validated_lines)

,po_number,customer_id,vendor_id,order_date,status,item_id,quantity_ordered,unit_price_paid,extended_amount
0,PO-100000,CUST-A,VEND-06,2024-04-01,Closed,ITM-0431,10,0.72,7.20
1,PO-100001,CUST-B,VEND-12,2026-05-21,Closed,ITM-0220,10,2.66,26.60
2,PO-100001,CUST-B,VEND-12,2026-05-21,Closed,ITM-0426,4,26.05,104.20
3,PO-100001,CUST-B,VEND-12,2026-05-21,Closed,ITM-0612,2,4.71,9.42
4,PO-100001,CUST-B,VEND-12,2026-05-21,Closed,ITM-1015,12,4.80,57.60
...,...,...,...,...,...,...,...,...,...
8470,PO-103179,CUST-D,VEND-08,2024-02-01,Received,ITM-0230,2,2.97,5.94
8471,PO-103179,CUST-D,VEND-08,2024-02-01,Received,ITM-0306,10,0.89,8.90
8472,PO-103179,CUST-D,VEND-08,2024-02-01,Received,ITM-0517,10,42.17,421.70
8473,PO-103179,CUST-D,VEND-08,2024-02-01,Received,ITM-1018,2,4.81,9.62
